# De tu audio a un video con letras
Ejecuta los seis pasos con ▶. No necesitas escribir código.

**Necesitas:** audio propio o autorizado y el paquete `.whl` recibido.

Selecciona **T4 GPU** en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Su disponibilidad depende de Colab.

**Iniciar → Tu canción → Preparar → Revisar → Generar → Descargar.**

El audio y los resultados permanecen en esta sesión temporal de Colab: descárgalos antes de cerrarla. No se monta Google Drive. El modelo de alineación tiene licencia CC-BY-NC-4.0; consulta los avisos de terceros del proyecto.

In [ ]:
# @title 1 · Iniciar
# @markdown Sube el paquete .whl. Se instalará el motor y se comprobará la GPU.
import json
import os
import shutil
import signal
import subprocess
import sys
import uuid
import zipfile
from pathlib import Path

import ipywidgets as widgets
from google.colab import files, output
from IPython.display import Video, display

if sys.version_info < (3, 12):  # noqa: UP036 - validate the host before installing the package
    raise RuntimeError('Selecciona un entorno de Colab con Python 3.12 o posterior.')
ROOT = Path('/content/karaoke-notebook')
ROOT.mkdir(exist_ok=True)
OUTPUT_DIR = ROOT / 'projects'
ENGINE = Path('/content/karaoke-env/bin/python')

def select_file(folder, extensions, label):
    folder.mkdir(parents=True, exist_ok=True)
    candidates = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in extensions]
    if len(candidates) != 1:
        print(label)
        uploaded = files.upload(target_dir=str(folder))
        candidates = [folder / Path(name).name for name in uploaded if Path(name).suffix.lower() in extensions]
    if len(candidates) != 1:
        raise ValueError('Selecciona exactamente un archivo del tipo indicado y repite este paso.')
    print('Archivo seleccionado:', candidates[0].name)
    return candidates[0]

def run_command(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            if process.wait():
                raise RuntimeError('El proceso no terminó correctamente. Revisa el mensaje anterior; no continúes.')
        except BaseException:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGINT)
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
            raise

def run_engine(operation, request):
    request_path = ROOT / f'{uuid.uuid4().hex}.request.json'
    result_path = request_path.with_suffix('.result.json')
    request_path.write_text(json.dumps(request), encoding='utf-8')
    run_command([str(ENGINE), '-u', '-m', 'karaoke_app.notebook_runner', operation,
                 '--request', str(request_path), '--result', str(result_path)])
    if not result_path.is_file():
        raise RuntimeError('No se recibió un resultado completo. Revisa los mensajes del proceso.')
    return json.loads(result_path.read_text(encoding='utf-8'))

wheel = select_file(ROOT / 'package', {'.whl'}, 'Sube el paquete .whl recibido.')
with zipfile.ZipFile(wheel) as package:
    requirements = package.read('karaoke_app/runtime/requirements.txt')
requirements_path = ROOT / 'requirements.txt'
requirements_path.write_bytes(requirements)
uv = shutil.which('uv')
if uv is None:
    bootstrap = ROOT / 'bootstrap'
    run_command([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap), 'uv==0.13.4'])
    uv = str(bootstrap / 'bin' / 'uv')
run_command([uv, 'venv', '--allow-existing', '--python', sys.executable, str(ENGINE.parent.parent)])
run_command([uv, 'pip', 'install', '--python', str(ENGINE), '--reinstall-package', 'lyrics-generator',
             str(wheel), '-r', str(requirements_path)])
run_command([uv, 'pip', 'check', '--python', str(ENGINE)])
print('Instalación completa. El entorno de Colab no se ha reemplazado.')

# Los controles viven en el Python de Colab; el motor sigue aislado.
output.enable_custom_widget_manager()

def create_song_form():
    """Create separate uploads and metadata controls for the current session."""
    fields = {
        'title': widgets.Text(description='Título', placeholder='Nombre real de la canción'),
        'artist': widgets.Text(description='Artista', placeholder='Intérprete de esta versión'),
        'album': widgets.Text(description='Álbum', placeholder='Opcional'),
        'language': widgets.Dropdown(description='Idioma', options=[
            ('Español', 'es'), ('English', 'en'), ('Português', 'pt'),
            ('Français', 'fr'), ('Italiano', 'it'), ('Deutsch', 'de')]),
        'search': widgets.Checkbox(value=True, description='Buscar letra en internet', indent=False),
    }
    for field in fields.values():
        field.layout = widgets.Layout(width='100%', max_width='520px')
        field.style.description_width = '70px'
    state = {'audio': None, 'reference': None, 'metadata': {}}
    lyrics_upload = widgets.FileUpload(
        accept='.txt', multiple=False, description='Seleccionar TXT', layout=widgets.Layout(width='210px'))
    lyrics_label = widgets.Label('Búsqueda automática')
    remove = widgets.Button(description='Quitar TXT', disabled=True)
    notice = widgets.Label('')

    def invalidate(change=None):
        globals().update(prepared=None, rendered=None)
        notice.value = 'Datos editables · continúa al paso 4' if state['audio'] else 'Selecciona un audio'
        if state['reference'] is None:
            lyrics_label.value = ('Búsqueda automática' if fields['search'].value else 'Transcripción')

    def receive(change, kind, control, label, extensions):
        if not change['new']:
            return
        try:
            value = change['new']
            if isinstance(value, dict):  # ipywidgets 7; la versión 8 entrega una tupla.
                entries = [dict(item, name=name) for name, item in value.items()]
            else:
                entries = list(value)
            if len(entries) != 1:
                raise ValueError('Elige exactamente un archivo en este botón.')
            item = entries[0]
            name = Path(item['name']).name
            if Path(name).suffix.lower() not in extensions:
                raise ValueError('Tipo incorrecto. Usa el botón de audio o el de TXT según corresponda.')
            if not len(item['content']):
                raise ValueError('El archivo está vacío. Selecciona otro.')
            folder = ROOT / 'uploads' / uuid.uuid4().hex
            folder.mkdir(parents=True)
            path = folder / name
            path.write_bytes(bytes(item['content']))
            state[kind] = path
            invalidate()
            label.value = f'✓ Archivo recibido: {name} · {path.stat().st_size / 1024 / 1024:.1f} MB'
            remove.disabled = state['reference'] is None
        except (OSError, ValueError, RuntimeError) as error:
            notice.value = f'{error} La selección anterior se conserva.'
        finally:
            # No conservar el contenido del archivo en el estado guardable del widget.
            control.value = {} if isinstance(control.value, dict) else ()

    lyrics_upload.observe(lambda change: receive(
        change, 'reference', lyrics_upload, lyrics_label, {'.txt'}), names='value')

    def clear_lyrics(button):
        state['reference'] = None
        remove.disabled = True
        invalidate()

    remove.on_click(clear_lyrics)
    for field in fields.values():
        field.observe(invalidate, names='value')
    reference_panel = widgets.VBox([
        widgets.HTML('<b>Aportar mi letra · opcional</b>'),
        widgets.HTML('<small>Se usará en lugar de la búsqueda automática.</small>'),
        lyrics_upload, lyrics_label, remove,
    ])
    panel = widgets.VBox([
        widgets.HTML('<br><b>Datos de la canción</b>'),
        fields['title'], fields['artist'], fields['album'], fields['language'],
        fields['search'], reference_panel, notice,
    ], layout=widgets.Layout(width='100%', max_width='600px', padding='12px', gap='8px'))
    return {'panel': panel, 'fields': fields, 'state': state}

song_form = create_song_form()
prepared = rendered = None
report = run_engine('diagnose', {})
if not report['cuda_available']:
    raise RuntimeError('No se detectó GPU. Selecciona T4 en el tipo de entorno y vuelve al paso 1.')
if not all(report['modules'].values()) or not all(report['commands'].values()):
    raise RuntimeError('Faltan componentes. Revisa la instalación del paso 1 antes de continuar.')
print('Motor listo · GPU:', report['gpu'])


In [ ]:
# @title 2 · Subir audio
# @markdown Selecciona un audio o vídeo. Esta celda espera hasta que termine la subida.
if 'song_form' not in globals():
    raise RuntimeError('Primero completa el paso 1.')
prepared = rendered = None
song_form['state'].update(audio=None, validated_audio=None)
song_form = create_song_form()
upload_dir = ROOT / 'uploads' / uuid.uuid4().hex
print('Selecciona un archivo y espera a que termine de subir…', flush=True)
uploaded = files.upload(target_dir=str(upload_dir))
uploaded_paths = [Path(name) for name in uploaded]
del uploaded
if len(uploaded_paths) != 1:
    raise ValueError('Selecciona exactamente un audio o vídeo. Vuelve a ejecutar este paso.')
audio_path = uploaded_paths[0]
if audio_path.suffix.lower() not in {'.wav', '.flac', '.mp3', '.m4a', '.ogg', '.mp4', '.webm'}:
    raise ValueError('Formato no admitido. Usa WAV, FLAC, MP3, M4A, OGG, MP4 o WEBM.')
if not audio_path.is_file() or not audio_path.stat().st_size:
    raise ValueError('No se recibió un archivo con contenido. Vuelve a ejecutar este paso.')
song_form['state']['audio'] = audio_path
print(f'✓ Archivo recibido: {audio_path.name} · {audio_path.stat().st_size / 1024 / 1024:.1f} MB')
print('Subida terminada. Ejecuta el paso 3 · Datos de la canción.')


In [ ]:
# @title 3 · Datos de la canción
# @markdown Lee los metadatos del audio. Confirma o completa los campos y el idioma.
if 'song_form' not in globals() or song_form['state']['audio'] is None:
    raise RuntimeError('Primero sube un audio en el paso 2 y espera la confirmación.')
state = song_form['state']
state['validated_audio'] = None
print('Leyendo metadatos…', flush=True)
selection = run_engine('validate-input', {'paths': [str(state['audio'])]})
metadata = selection.get('metadata', {})
for key, label in [('title', 'Título'), ('artist', 'Artista'), ('album', 'Álbum')]:
    field = song_form['fields'][key]
    if not field.value.strip() or field.value == state['metadata'].get(key):
        field.value = metadata.get(key) or ''
    print(f'{label} detectado: {metadata.get(key) or "Sin etiqueta; puedes completarlo manualmente"}')
state['metadata'] = metadata
state['validated_audio'] = state['audio']
display(song_form['panel'])


In [ ]:
# @title 4 · Crear voces y letra sincronizada
# @markdown Genera el proyecto editable, todavía no el vídeo. La primera vez descarga modelos.
# @markdown **Modelo Whisper:** small (rápido), medium (equilibrado), large-v3 (más preciso).
Modelo = 'large-v3' # @param ['small', 'medium', 'large-v3']
prepared = None
rendered = None
if 'song_form' not in globals() or song_form['state']['audio'] is None:
    raise RuntimeError('Selecciona un audio en el paso 2.')
if song_form['state'].get('validated_audio') != song_form['state']['audio']:
    raise RuntimeError('Ejecuta el paso 3 para leer y confirmar los datos del audio actual.')
fields = song_form['fields']
audio = song_form['state']['audio']
reference = song_form['state']['reference']
TITULO = fields['title'].value
ARTISTA = fields['artist'].value
ALBUM = fields['album'].value
IDIOMA = fields['language'].value
BUSCAR_LETRA_EN_LINEA = fields['search'].value
if BUSCAR_LETRA_EN_LINEA and reference is None and (not TITULO.strip() or not ARTISTA.strip()):
    raise ValueError('Completa título y artista en el formulario para buscar la letra automáticamente.')
print('Modelo Whisper:', Modelo, flush=True)
print('Validando audio…', flush=True)
selection = run_engine('validate-input', {'paths': [str(audio)] + ([str(reference)] if reference else [])})
media_info = selection['media']
print(f"Audio validado: {audio.name} · {media_info['duration_seconds']:.1f} segundos")
print('Letra:', reference.name if reference else ('Búsqueda en línea' if BUSCAR_LETRA_EN_LINEA else 'Transcripción'))
prepared = run_engine('prepare', {
    'source': str(audio), 'output_dir': str(OUTPUT_DIR), 'title': TITULO.strip() or None,
    'artist': ARTISTA.strip() or None, 'album': ALBUM.strip() or None, 'language': IDIOMA,
    'lyrics_text': str(reference) if reference else None, 'use_lrclib': BUSCAR_LETRA_EN_LINEA,
    'whisper_model': Modelo,
})
print('Proyecto preparado. Continúa al paso 5 o ejecuta la celda opcional 4b para revisar la letra.')
info = run_engine('inspect', {'directory': prepared['directory']})
evidence = info['review']
sources = {'lrclib': 'LRCLIB', 'genius': 'Genius', 'txt': 'TXT aportado',
           'whisper': 'Transcripción automática (Whisper)'}
print('Origen de la letra:', sources.get(evidence['reference_source'], 'Sin información'))
if evidence['reference_source'] == 'whisper':
    print('La transcripción automática puede contener errores.')
if evidence['needs_review'] or evidence['anchor_status'] == 'insufficient_anchors':
    print('Tiempos pendientes de revisión. Puedes revisarlos en LyricsEditor desde la celda 4b.')


In [ ]:
# @title 4b · Revisar en LyricsEditor (opcional)
# @markdown [Abrir LyricsEditor ↗](https://dotslashgabut.github.io/lyricseditor/)
# @markdown Descarga las voces y la letra ejecutando una acción cada vez. Abre ambos archivos en el editor.
# @markdown Para aplicar cambios, exporta LyricsFile YAML y elige **Importar YAML corregido**. No repite la preparación.
Accion = 'Descargar voces' # @param ['Descargar voces', 'Descargar letra', 'Importar YAML corregido']
if not globals().get('prepared'):
    raise RuntimeError('Primero completa el paso 4.')
project = prepared['directory']
print('Editor: https://dotslashgabut.github.io/lyricseditor/')
if Accion == 'Importar YAML corregido':
    folder = ROOT / 'reviews' / uuid.uuid4().hex
    uploaded = files.upload(target_dir=str(folder))
    paths = [Path(name) for name in uploaded]
    del uploaded
    if len(paths) != 1 or paths[0].suffix.lower() not in {'.yaml', '.yml'}:
        raise ValueError('Selecciona exactamente un archivo LyricsFile YAML exportado por el editor.')
    if not globals().get('prepared') or prepared['directory'] != project:
        raise RuntimeError('La canción cambió. Repite esta celda para el proyecto actual.')
    run_engine('import-lyrics', {'directory': project, 'source': str(paths[0])})
    rendered = None
    print('✓ YAML actualizado. El paso 5 generará el vídeo con la letra corregida.')
else:
    if Accion == 'Descargar voces':
        info = run_engine('inspect', {'directory': project})
        filename = Path(info['vocals'])
    elif Accion == 'Descargar letra':
        filename = Path(project) / 'lyrics.lyricsfile.yaml'
    else:
        raise ValueError('Selecciona una acción del formulario.')
    if not filename.is_file():
        raise FileNotFoundError(f'Archivo no disponible: {filename.name}')
    print(f'Descargando: {filename.name}')
    files.download(str(filename))


In [ ]:
# @title 5 · Generar el vídeo
# @markdown Usa la letra actual. Cambiar el aspecto no repite la preparación.
Modo = 'Lyrics' # @param ['Lyrics', 'Karaoke', 'Ambos']
Estilo = 'Clásico' # @param ['Clásico', 'Fluido', 'Cósmico']
Calidad = '720p' # @param ['720p', '1080p']
if not globals().get('prepared'):
    raise RuntimeError('Primero completa el paso 4.')
rendered = None
rendered = run_engine('render', {'directory': prepared['directory'], 'options': {
    'mode': {'Lyrics': 'lyrics', 'Karaoke': 'karaoke', 'Ambos': 'both'}[Modo],
    'preset': {'Clásico': 'classic', 'Fluido': 'fluid', 'Cósmico': 'cosmic'}[Estilo],
    'resolution': Calidad,
}})
print('Video terminado. Puedes verlo y descargarlo a continuación.')
for video in rendered['videos']:
    print(Path(video).name)
    display(Video(video, embed=True, width=640))


In [ ]:
# @title 6 · Descargar
Descargar = 'Vídeos (MP4)' # @param ['Vídeos (MP4)', 'Vídeos (ZIP)', 'Proyecto editable (ZIP)']
if Descargar == 'Proyecto editable (ZIP)':
    if not globals().get('prepared'):
        raise RuntimeError('Primero completa el paso 4.')
    exported = run_engine('export-project', {'directory': prepared['directory']})
    downloads = [exported['archive']]
else:
    if not globals().get('rendered'):
        raise RuntimeError('Primero completa el paso 5.')
    downloads = rendered['videos'] if Descargar == 'Vídeos (MP4)' else [rendered['archive']]
for filename in downloads:
    print('Descargando:', Path(filename).name)
    files.download(filename)
print('Conserva también el proyecto editable si quieres volver a trabajar con esta canción.')


## Al terminar
Comprueba que las descargas se abren en tu equipo. Si el navegador bloquea varias descargas, elige el paquete ZIP. Después usa **Entorno de ejecución → Desconectar y eliminar entorno de ejecución** para liberar la GPU.

**Si falla un paso:** no continúes con los siguientes. Lee el último mensaje; los intentos fallidos conservan diagnósticos en la ruta indicada. Si pierdes la sesión, vuelve al paso 1. Los archivos que no descargues se pueden perder.

**Para desarrolladores:** el cuaderno utiliza `karaoke_app.notebook_runner` como adaptador del mismo motor de preparación y render que utiliza la CLI. No abre otra sesión de Colab ni modifica el Python del proveedor. Consulta `docs/notebooks.md` para distribución, arquitectura y resultados de verificación.